# Логистическая регрессия

Практические ячейки к слайдам презентации. Заголовки секций совпадают с заголовками слайдов.


In [ ]:
# Setup
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

np.random.seed(42)



## Почему линейная регрессия не подходит? `(example)`


Закодируем классы как 0 и 1 и обучим обычный `LinearRegression` — предсказания выходят за [0, 1].




In [ ]:
X, y = make_classification(n_samples=60, n_features=1, n_informative=1,
                           n_redundant=0, n_clusters_per_class=1,
                           class_sep=1.5, random_state=42)

lin = LinearRegression().fit(X, y)
y_pred = lin.predict(X)

print(f'Мин. predict: {y_pred.min():.3f}')
print(f'Макс. predict: {y_pred.max():.3f}')
print('Значения вне [0, 1]:', np.sum((y_pred < 0) | (y_pred > 1)), 'из', len(y_pred))

plt.figure(figsize=(7, 4))
plt.scatter(X[y == 0], y[y == 0], label='класс 0', alpha=0.8)
plt.scatter(X[y == 1], y[y == 1], label='класс 1', alpha=0.8)
x_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
plt.plot(x_line, lin.predict(x_line), 'r-', linewidth=2, label='LinearRegression')
plt.axhline(0, color='gray', linestyle='--', alpha=0.6)
plt.axhline(1, color='gray', linestyle='--', alpha=0.6)
plt.xlabel('признак x')
plt.ylabel('метка / predict')
plt.title('МНК на бинарных метках: predict вне [0, 1]')
plt.legend()
plt.tight_layout()
plt.show()


## Почему линейная регрессия не подходит? `(experiment)`


Один выброс сильно наклоняет линию МНК — модель не устойчива к аномалиям.




In [ ]:
X_base, y_base = make_classification(n_samples=50, n_features=1, n_informative=1,
                                       n_redundant=0, n_clusters_per_class=1,
                                       class_sep=1.5, random_state=42)

X_out = np.vstack([X_base, [[X_base.max() + 0.5]]])
y_out = np.append(y_base, 1)

model_base = LinearRegression().fit(X_base, y_base)
model_out = LinearRegression().fit(X_out, y_out)

x_plot = np.linspace(X_out.min(), X_out.max(), 100).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, X_, y_, model, title in [
    (axes[0], X_base, y_base, model_base, 'Без выброса'),
    (axes[1], X_out, y_out, model_out, 'С выбросом'),
]:
    ax.scatter(X_[y_ == 0], y_[y_ == 0], label='класс 0')
    ax.scatter(X_[y_ == 1], y_[y_ == 1], label='класс 1')
    ax.plot(x_plot, model.predict(x_plot), 'r-', linewidth=2)
    if title == 'С выбросом':
        ax.scatter(X_out[-1], 1, color='orange', s=120, zorder=5, label='выброс')
        ax.legend()
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y / predict')
plt.tight_layout()
plt.show()

print(f'Наклон без выброса: {model_base.coef_[0]:.3f}')
print(f'Наклон с выбросом:  {model_out.coef_[0]:.3f}')


## Сигмоида: сжатие выхода в [0, 1] `(viz)`


Сигмоида $\sigma(z) = 1/(1+e^{-z})$ сжимает любое $z$ в интервал $(0, 1)$ — это и есть вероятность класса 1.




In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-6, 6, 300)

plt.figure(figsize=(7, 4))
plt.plot(z, sigmoid(z), 'crimson', linewidth=2)
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='порог 0.5')
plt.axvline(0, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('z = w·x + b')
plt.ylabel('σ(z) = P(y=1|x)')
plt.title('Сигмоида: S-образная кривая')
plt.ylim(-0.05, 1.05)
plt.legend()
plt.tight_layout()
plt.show()

print(f'σ(-6) ≈ {sigmoid(-6):.4f}, σ(0) = {sigmoid(0):.2f}, σ(6) ≈ {sigmoid(6):.4f}')


## LogLoss (бинарная кросс-энтропия) `(viz)`


LogLoss сильно штрафует уверенные ошибки: при $y=1$ штраф растёт, когда $p \to 0$.




In [ ]:
def logloss_y1(p):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return -np.log(p)

def logloss_y0(p):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return -np.log(1 - p)

p = np.linspace(0.01, 0.99, 200)

plt.figure(figsize=(7, 4))
plt.plot(p, logloss_y1(p), label='y = 1: −log(p)', linewidth=2)
plt.plot(p, logloss_y0(p), label='y = 0: −log(1−p)', linewidth=2)
plt.xlabel('предсказанная вероятность p')
plt.ylabel('штраф LogLoss')
plt.title('LogLoss: штраф за уверенную ошибку огромен')
plt.legend()
plt.tight_layout()
plt.show()

for p_val in [0.99, 0.5, 0.01]:
    print(f'y=1, p={p_val}: штраф = {logloss_y1(p_val):.3f}')


## Обучение: градиент и оптимизация `(example)`


LogReg в sklearn сходится итеративно (L-BFGS). Сравним loss на train при разном `max_iter`.




In [ ]:
X, y = make_classification(n_samples=200, n_features=4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

for max_iter in [10, 50, 500]:
    pipe = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=max_iter, random_state=42)
    )
    pipe.fit(X_train, y_train)
    lr = pipe.named_steps['logisticregression']
    print(f'max_iter={max_iter:3d}: n_iter_={lr.n_iter_[0]:3d}, '
          f'accuracy test={pipe.score(X_test, y_test):.3f}')


## ROC, AUC-ROC и PR-кривая `(example)`


ROC показывает TPR vs FPR при всех порогах; PR-кривая важнее при сильном дисбалансе классов.




In [ ]:
X, y = make_classification(n_samples=500, n_features=4, weights=[0.9, 0.1],
                           random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, random_state=42))
pipe.fit(X_train, y_train)
proba = pipe.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[0])
axes[0].set_title('ROC-кривая')
PrecisionRecallDisplay.from_predictions(y_test, proba, ax=axes[1])
axes[1].set_title('PR-кривая (дисбаланс 90/10)')
plt.tight_layout()
plt.show()


## Логистическая регрессия в sklearn `(example)`


Pipeline с `StandardScaler` обязател: регуляризация чувствительна к масштабу. Главное — `predict_proba`, не только `predict`.




In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=1000, random_state=42)
)
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:5, 1]
pred = model.predict(X_test)[:5]

print('Первые 5 объектов test:')
for i in range(5):
    print(f'  P(y=1)={proba[i]:.3f} → predict={pred[i]} (истина {y_test[i]})')
print(f'\nAccuracy на test: {model.score(X_test, y_test):.3f}')


## Несбалансированные данные на практике `(experiment)`


При дисбалансе модель без `class_weight` может игнорировать минорный класс. `class_weight='balanced'` усиливает штраф за ошибки на редком классе.




In [ ]:
X, y = make_classification(n_samples=1000, n_features=6, weights=[0.97, 0.03],
                           random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

for cw, label in [(None, 'без class_weight'), ('balanced', 'class_weight=balanced')]:
    pipe = make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight=cw, max_iter=500, random_state=42)
    )
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    print(f'--- {label} ---')
    print(classification_report(y_test, y_pred, digits=3))
    print()


## Калибровка вероятностей `(viz)`


Reliability diagram: если модель предсказала 0.7 для группы объектов, около 70% из них должны быть классом 1. LogReg обычно хорошо калибрована.




In [ ]:
X, y = make_classification(n_samples=2000, n_features=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42)

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, random_state=42))
pipe.fit(X_train, y_train)
proba = pipe.predict_proba(X_test)[:, 1]

frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=10, strategy='uniform')

plt.figure(figsize=(6, 5))
plt.plot([0, 1], [0, 1], 'k--', label='идеальная калибровка')
plt.plot(mean_pred, frac_pos, 'o-', color='crimson', label='LogReg')
plt.xlabel('средняя предсказанная вероятность')
plt.ylabel('доля положительных (факт)')
plt.title('Reliability diagram')
plt.legend()
plt.tight_layout()
plt.show()


## Ограничения и связь с нейросетями `(viz)`


XOR — классический пример, где линейная граница не работает. Один нейрон с сигмоидой = логистическая регрессия.




In [ ]:
X, y = make_classification(n_samples=120, n_features=2, n_redundant=0,
                           n_informative=2, n_clusters_per_class=1,
                           class_sep=0.8, flip_y=0.08, random_state=42)
# XOR-подобная структура
y_xor = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).astype(int)

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, random_state=42))
pipe.fit(X, y_xor)

xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200),
    np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 200)
)
Z = pipe.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, Z, levels=20, cmap='RdBu', alpha=0.6)
plt.colorbar(label='P(y=1)')
plt.scatter(X[y_xor == 0, 0], X[y_xor == 0, 1], c='blue', edgecolors='k', label='класс 0')
plt.scatter(X[y_xor == 1, 0], X[y_xor == 1, 1], c='red', edgecolors='k', label='класс 1')
plt.title('XOR: линейная граница не разделяет классы')
plt.xlabel('x₁')
plt.ylabel('x₂')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Accuracy на XOR: {pipe.score(X, y_xor):.3f}')
